# PINN Vancomycin — Sensitivity Analysis (Google Colab)

Notebook za pokretanje glavnog eksperimenta na Colab GPU/CPU.

**Redosljed pokretanja:** izvrsavaj ćelije redom odozgo prema dolje.  
**Checkpoint:** svaki run se upisuje u Google Drive — prekid sesije ne gubi napredak.  
**Nastavak:** ako se sesija prekine, pokreni ćelije 1–5 ponovo, pa `Restore checkpoint` ćeliju, pa nastavi od mjesta gdje si stala.

---
| Ćelija | Akcija | Trajanje |
|--------|--------|----------|
| 1 | Mount Drive + clone repo | 1 min |
| 2 | Instalacija zavisnosti | 2 min |
| 3 | Generisanje sintetičkih podataka | 30 s |
| 4 | Setup i import modula | 5 s |
| 5 | Test run (provjera da sve radi) | ~3 min |
| 6 | **1-odeljni model — puni eksperiment** | ~2–5h |
| 7 | Preuzimanje 1-comp rezultata | 5 s |
| 8 | **2-odeljni model — puni eksperiment** | ~8–15h |
| 9 | Preuzimanje 2-comp rezultata + zip | 5 s |

## Ćelija 1 — Google Drive + kloniranje repoa

⚠️ **Izmijeni `REPO_URL`** na stvarni GitHub link prije pokretanja.

In [ ]:
# ── Google Drive ──────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/pinn_vancomycin')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive folder: {DRIVE_DIR}')

# ── Kloniranje repoa ──────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'   # <-- IZMIJENI OVO
REPO_DIR = Path('/content/pinn-vancomycin')

if not REPO_DIR.exists():
    !git clone $REPO_URL {REPO_DIR}
else:
    print('Repo vec postoji, povlacim promjene...')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!git log --oneline -3

## Ćelija 2 — Instalacija zavisnosti

In [ ]:
# torch je vec instaliran na Colabu; ostale biblioteke instaliramo tise
!pip install scipy numpy pandas matplotlib seaborn -q

import torch
print(f'PyTorch verzija: {torch.__version__}')
print(f'CUDA dostupna:   {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')

## Ćelija 3 — Generisanje sintetičkih podataka

Kreira 4 CSV fajla u `data/processed/`. Traje ~30s.

In [ ]:
!python src/data_processing.py

## Ćelija 4 — Setup: putanje, device, import modula

In [ ]:
import sys
import importlib.util
import torch
from pathlib import Path

REPO_DIR  = Path('/content/pinn-vancomycin')
DRIVE_DIR = Path('/content/drive/MyDrive/pinn_vancomycin')

sys.path.insert(0, str(REPO_DIR / 'src'))

# Ucitaj 03_sensitivity_analysis.py kao modul
# (naziv pocinje cifrom pa ne moze direktno import)
def _load(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

sens = _load(REPO_DIR / 'experiments' / '03_sensitivity_analysis.py', 'sensitivity')

# Odabir device-a
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Checkpoint folder: {DRIVE_DIR}')
print('Moduli ucitani.')

## Ćelija 5 — Test run (opcionalno, preporučeno pri prvom pokretanju)

Pokreće 8 kombinacija (N∈{3,5}, σ∈{0.0,0.05}, 2 seed-a) sa minimalnim brojem epoha.  
Trajanje: ~3–5 min. Ako prođe bez greške, sve je ispravno.

In [ ]:
print('=== TEST RUN — 1-odeljni ===')
sens.run_sensitivity_1comp(test_mode=True, out_dir=DRIVE_DIR, device=DEVICE)

print()
print('=== TEST RUN — 2-odeljni ===')
sens.run_sensitivity_2comp(test_mode=True, out_dir=DRIVE_DIR, device=DEVICE)

print()
print('Test zavrsen. Obrisi test CSV-ove iz Drive foldera prije pravog eksperimenta:')
print('  sensitivity_1comp.csv  i  sensitivity_2comp.csv')

In [ ]:
# Obrisi test CSV-ove da ne kontaminiraju pravi eksperiment
import os
for name in ['sensitivity_1comp.csv', 'sensitivity_2comp.csv']:
    p = DRIVE_DIR / name
    if p.exists():
        os.remove(p)
        print(f'Obrisan: {p}')
print('Spreman za pravi eksperiment.')

## Ćelija 6 — 1-odeljni model: puni eksperiment

**600 runova** (N×σ×30 seed-ova). Svaki run se odmah upisuje u Drive.  
Procijenjeno trajanje: **~2–5h** (CPU) / **~1–2h** (GPU T4).

> Checkpoint je automatski: ako se sesija prekine, pokreni ćelije 1–4 i `Restore checkpoint` ćeliju, pa ponovi ovu ćeliju — preskočiće već završene kombinacije.

In [ ]:
sens.run_sensitivity_1comp(test_mode=False, out_dir=DRIVE_DIR, device=DEVICE)

## Ćelija 7 — Preuzimanje 1-comp rezultata na lokalni računar

In [ ]:
from google.colab import files
import shutil

src = DRIVE_DIR / 'sensitivity_1comp.csv'
if src.exists():
    shutil.copy(src, '/content/sensitivity_1comp.csv')
    files.download('/content/sensitivity_1comp.csv')
    print(f'Preuzimanje pokrenuto ({src.stat().st_size // 1024} KB)')
else:
    print('CSV nije pronadjen. Provjeri DRIVE_DIR.')

## Ćelija 8 — 2-odeljni model: puni eksperiment

**600 runova**. Procijenjeno trajanje: **~8–15h** (CPU) / **~3–5h** (GPU T4).  

> Zbog duzine, preporucujem pokrenuti u Colab Pro sesiji ili koristiti GPU runtime (Runtime → Change runtime type → T4 GPU).

In [ ]:
sens.run_sensitivity_2comp(test_mode=False, out_dir=DRIVE_DIR, device=DEVICE)

## Ćelija 9 — Preuzimanje svih rezultata (zip)

In [ ]:
import shutil
from google.colab import files

# Preuzmi 2-comp CSV
src2 = DRIVE_DIR / 'sensitivity_2comp.csv'
if src2.exists():
    shutil.copy(src2, '/content/sensitivity_2comp.csv')
    files.download('/content/sensitivity_2comp.csv')
    print(f'2-comp preuzet ({src2.stat().st_size // 1024} KB)')

# Zip svih CSV-ova iz Drive foldera
zip_path = '/content/pinn_results_all'
shutil.make_archive(zip_path, 'zip', DRIVE_DIR)
files.download(zip_path + '.zip')
print('Zip svih rezultata preuzet.')

---
## Restore checkpoint (koristiti samo ako se sesija prekinula)

Pokreni ovu ćeliju **nakon** ćelija 1–4, a **prije** ponovnog pokretanja eksperimenta.  
Kopira checkpoint CSV-ove iz Drive-a u lokalni `results/tables/` folder.

In [ ]:
import shutil
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/pinn_vancomycin')
LOCAL_TAB = Path('/content/pinn-vancomycin/results/tables')
LOCAL_TAB.mkdir(parents=True, exist_ok=True)

restored = 0
for csv in sorted(DRIVE_DIR.glob('sensitivity_*.csv')):
    dst  = LOCAL_TAB / csv.name
    shutil.copy(csv, dst)
    rows = sum(1 for _ in open(dst)) - 1   # bez headera
    print(f'  Restored: {csv.name}  ({rows} redova)')
    restored += 1

if restored == 0:
    print('Nema checkpoint fajlova u Drive folderu.')
else:
    print(f'\nRestore zavrsen. Nastavi sa celijom 6 ili 8.')

---
## Brza provjera napretka

Pokreni u bilo kom trenutku da vidis koliko je redova upisano.

In [ ]:
import pandas as pd
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/pinn_vancomycin')

for name in ['sensitivity_1comp.csv', 'sensitivity_2comp.csv']:
    p = DRIVE_DIR / name
    if not p.exists():
        print(f'{name}: ne postoji')
        continue
    df   = pd.read_csv(p)
    total = 5 * 4 * 30   # N x sigma x seed
    print(f'{name}: {len(df)}/{total} redova ({len(df)/total*100:.1f}%)')
    # Distribucija po N
    if len(df) > 0:
        print('  Po N:', df['N'].value_counts().sort_index().to_dict())
        # Provjeri greske
        if 'pinn_err_k10' in df.columns:
            ok = df[df['status'] == 'ok']
            print(f'  Uspjesnih runova: {len(ok)}/{len(df)}')
            print(f'  Median PINN err_k10: {ok["pinn_err_k10"].median():.2f}%')
            print(f'  Median bench err_k10: {ok["bench_err_k10"].median():.2f}%')
    print()